# ⚡ Samtidiga Agentarbetsflöden med GitHub-modeller (Python)

## 📋 Avancerad tutorial för parallell bearbetning

Denna anteckningsbok demonstrerar **samtidiga arbetsflödesmönster** med Microsoft Agent Framework. Du kommer att lära dig hur man bygger högpresterande, parallella arbetsflöden där flera AI-agenter körs samtidigt, vilket dramatiskt förbättrar genomströmningen och möjliggör sofistikerade flertrådade affärsprocesser.

## 🎯 Lärandemål

### 🚀 **Grundläggande samtida bearbetning**
- **Parallell agentkörning**: Kör flera agenter samtidigt för maximal effektivitet
- **Arbetsflödesorkestrering**: Koordinera samtidiga operationer samtidigt som datakonsistens upprätthålls
- **Prestandaoptimering**: Uppnå betydande snabbhet genom parallell bearbetning
- **Resurshantering**: Effektiv användning av AI-modellresurser i samtidiga operationer

### 🏗️ **Avancerade samtidighetsmönster**
- **Fork-Join-bearbetning**: Dela upp arbete mellan flera agenter och slå samman resultat
- **Pipelinedrift**: Överlappande exekveringssteg för kontinuerlig genomströmning
- **Lastbalansering**: Fördela arbete jämnt över tillgängliga agentresurser
- **Synkroniseringspunkter**: Koordinera samtidiga agenter i kritiska arbetsflödesskeden

### 🏢 **Samtida företagsapplikationer**
- **Högvolymdokumentbearbetning**: Bearbeta flera dokument samtidigt
- **Realtime-innehållsanalys**: Samtidig analys av inkommande dataströmmar
- **Batchbearbetningsoptimering**: Maximera genomströmningen för storskaliga operationer
- **Multimodal analys**: Parallell bearbetning av olika innehållstyper (text, bilder, data)

## ⚙️ Förutsättningar och installation

### 📦 **Nödvändiga beroenden**

Installera Agent Framework med kapacitet för samtidiga arbetsflöden:

```bash
pip install agent-framework-core -U
```

### 🔑 **Konfiguration av GitHub-modeller**

**Miljöinställning (.env-fil):**
```env
GITHUB_TOKEN=your_github_personal_access_token
GITHUB_ENDPOINT=https://models.inference.ai.azure.com
GITHUB_MODEL_ID=gpt-4o-mini
```

**Överväganden för samtida bearbetning:**
- **Hastighetsbegränsningar**: Övervaka GitHub Models API:s hastighetsbegränsningar för samtidiga förfrågningar
- **Resursanvändning**: Ta hänsyn till minnes- och CPU-användning med flera samtidiga agenter
- **Felhantering**: Implementera robust felåterställning för parallella operationer

### 🏗️ **Samtida arbetsflödesarkitektur**

```mermaid
graph TD
    A[Start av arbetsflöde] --> B[Parallell körning]
    B --> C[Agentpool 1]
    B --> D[Agentpool 2]
    B --> E[Agentpool 3]
    C --> F[Resultatsammanställning]
    D --> F
    E --> F
    F --> G[Slutlig utdata]
    
    H[GitHub Modeller API] --> C
    H --> D
    H --> E
```

**Viktiga fördelar:**
- **⚡ Prestanda**: Betydande snabbare genomförande genom parallell exekvering
- **📈 Skalbarhet**: Hantera ökade arbetsbelastningar utan proportionell tidsökning
- **🔄 Effektivitet**: Bättre utnyttjande av tillgängliga beräkningsresurser
- **🎯 Genomströmning**: Bearbeta mer arbete på samma tid

## 🎨 **Designmönster för samtidiga arbetsflöden**

### 🔍 **Forsknings- och analyspipeline**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Datahanteringsarbetsflöde**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Innehållsskapandepipeline**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Flerstegsprocessning**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Företagsprestandafördelar**

### ⚡ **Genomströmningsoptimering**
- **Parallell exekvering**: Flera agenter som arbetar samtidigt
- **Resursutnyttjande**: Maximal effektivitet av tillgänglig AI-modellkapacitet
- **Tidsminskning**: Betydande minskning av total bearbetningstid
- **Skalbar arkitektur**: Enkel att lägga till fler samtidiga agenter vid behov

### 🛡️ **Tillförlitlighet och motståndskraft**
- **Felmotstånd**: Fel hos enskilda agenter stoppar inte hela arbetsflödet
- **Felisolering**: Problem i en samtidig gren påverkar inte andra
- **Graceful degradation**: Systemet fortsätter fungera även med reducerad agentkapacitet
- **Återhämtningsmekanismer**: Automatisk omförsök och felhantering för misslyckade operationer

### 📊 **Övervakning och observerbarhet**
- **Spårning av samtidig exekvering**: Följ alla parallella operationers framsteg
- **Prestandamått**: Mät snabbhets- och effektivitetstillväxt
- **Analys av resursanvändning**: Optimera tilldelningen av samtidiga agenter
- **Identifiering av flaskhalsar**: Hitta och åtgärda prestandabegränsningar

Låt oss bygga högpresterande samtida AI-arbetsflöden! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = await provider.create_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = await provider.create_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)

In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
